In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')


FileNotFoundError: [Errno 2] No such file or directory: '../data/train.csv'

In [31]:
# 1 
counts = df['answer'].value_counts()
sum_most_least = counts.max() + counts.min()
sum_most_least


np.int64(814)


In [32]:
# 2
import string

def clean_text(text):
    return "".join(char for char in text.lower() if char not in string.punctuation)


In [33]:
vocab = set(" ".join(df['prompt'].apply(clean_text)).split())
len(vocab)


859


In [34]:
# 4
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

row1_tokens = clean_text(df.iloc[0]['prompt']).split()
filtered_tokens = [w for w in row1_tokens if w not in ENGLISH_STOP_WORDS]
len(filtered_tokens)


13


In [35]:
#5
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text = df['prompt'] + " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(combined_text)
len(vectorizer.get_feature_names_out())


2762


In [36]:
#6
prompt_vector = vectorizer.transform([df.iloc[0]['prompt']])
a_vector = vectorizer.transform([df.iloc[0]['A']])
cosine_similarity(prompt_vector, a_vector)


array([[0.2720243]])


In [37]:
def get_best_option(row):
    options = ['A', 'B', 'C', 'D', 'E']
    prompt_vector = vectorizer.transform([row['prompt']])
    scores = {}
    for option in options:
        option_vector = vectorizer.transform([row[option]])
        score = cosine_similarity(prompt_vector, option_vector)[0][0]
        scores[option] = score
    return max(scores, key=scores.get)

matches = df.apply(lambda row: get_best_option(row) == row['answer'], axis=1)
matches.mean() * 100


np.float64(13.55)


In [38]:
def map_at_3(truth, predictions):
    if truth in predictions[:3]:
        rank = predictions.index(truth) + 1
        return 1 / rank
    return 0

map_at_3('C', ['C', 'A', 'B']), map_at_3('B', ['D', 'B', 'E'])


(1.0, 0.5)


In [39]:
# 9
top3 = df.answer.value_counts().index.tolist()[:3]

scores = df['answer'].apply(lambda truth: map_at_3(truth, top3))
scores.mean()


np.float64(0.42125)


In [40]:
def top3_preds(row):
    options = ['A', 'B', 'C', 'D', 'E']
    prompt_vector = vectorizer.transform([row['prompt']])
    scores = {}
    for opt in options:
        opt_vector = vectorizer.transform([row[opt]])
        score = cosine_similarity(prompt_vector, opt_vector)[0][0]
        scores[opt] = score
    return sorted(scores, key=scores.get, reverse=True)[:3]

all_scores = df.apply(lambda row: map_at_3(row['answer'], top3_preds(row)), axis=1)
all_scores.mean()


np.float64(0.2961666666666667)
